# Лабораторна робота №2 — Частина 2
## Наука про дані: підготовчий етап
### Individual Household Electric Power Consumption Dataset

## 1. Імпорт бібліотек

In [1]:
import pandas as pd
import numpy as np
import timeit
import zipfile
import urllib.request
import os

## 2. Завантаження датасету

Завантажуємо Individual Household Electric Power Consumption Dataset з UCI Machine Learning Repository.

**Опис стовпців:**
- **Date** — дата (dd/mm/yyyy)
- **Time** — час (hh:mm:ss)
- **Global_active_power** — загальна активна потужність (кВт)
- **Global_reactive_power** — загальна реактивна потужність (кВт)
- **Voltage** — напруга (В)
- **Global_intensity** — сила струму (А)
- **Sub_metering_1** — кухня (посудомийка, духовка, мікрохвильовка) (Вт·год)
- **Sub_metering_2** — пральна машина, сушарка, холодильник, освітлення (Вт·год)
- **Sub_metering_3** — бойлер та кондиціонер (Вт·год)

In [2]:
# Завантаження датасету
DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip"
ZIP_FILE = "data/household_power_consumption.zip"
TXT_FILE = "data/household_power_consumption.txt"

os.makedirs("data", exist_ok=True)

if not os.path.exists(TXT_FILE):
    if not os.path.exists(ZIP_FILE):
        print("Завантажуємо датасет...")
        urllib.request.urlretrieve(DATA_URL, ZIP_FILE)
        print("Завантажено!")
    
    print("Розпаковуємо...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as z:
        z.extractall("data/")
    print("Розпаковано!")
else:
    print("Датасет вже завантажений")

print(f"Розмір файлу: {os.path.getsize(TXT_FILE) / 1024 / 1024:.1f} MB")

Завантажуємо датасет...
Завантажено!
Розпаковуємо...
Розпаковано!
Розмір файлу: 126.8 MB


## 3. Зчитування та очищення даних (Data Cleaning)

Зчитуємо датасет у pandas DataFrame. Обробляємо пропуски (позначені як '?'), конвертуємо типи даних.

In [3]:
# Зчитування з профілюванням часу
start = timeit.default_timer()

df = pd.read_csv(
    TXT_FILE,
    sep=';',
    low_memory=False,
    na_values=['?', '']
)

elapsed = timeit.default_timer() - start
print(f"Час зчитування: {elapsed:.2f} секунд")
print(f"Розмір DataFrame: {df.shape[0]} рядків × {df.shape[1]} стовпців")
print(f"\nПерші 5 записів:")
df.head()

Час зчитування: 9.45 секунд
Розмір DataFrame: 2075259 рядків × 9 стовпців

Перші 5 записів:


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


In [4]:
# Інформація про датасет
print("Інформація про DataFrame:")
print("=" * 60)
df.info()

Інформація про DataFrame:
<class 'pandas.DataFrame'>
RangeIndex: 2075259 entries, 0 to 2075258
Data columns (total 9 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Date                   str    
 1   Time                   str    
 2   Global_active_power    float64
 3   Global_reactive_power  float64
 4   Voltage                float64
 5   Global_intensity       float64
 6   Sub_metering_1         float64
 7   Sub_metering_2         float64
 8   Sub_metering_3         float64
dtypes: float64(7), str(2)
memory usage: 176.0 MB


In [5]:
# Перевірка пропусків
print("Кількість пропусків по стовпцях:")
print("-" * 40)
missing = df.isnull().sum()
for col, count in missing.items():
    pct = count / len(df) * 100
    print(f"  {col:30s}: {count:6d} ({pct:.2f}%)")

print(f"\nВсього рядків з пропусками: {df.isnull().any(axis=1).sum()}")

Кількість пропусків по стовпцях:
----------------------------------------
  Date                          :      0 (0.00%)
  Time                          :      0 (0.00%)
  Global_active_power           :  25979 (1.25%)
  Global_reactive_power         :  25979 (1.25%)
  Voltage                       :  25979 (1.25%)
  Global_intensity              :  25979 (1.25%)
  Sub_metering_1                :  25979 (1.25%)
  Sub_metering_2                :  25979 (1.25%)
  Sub_metering_3                :  25979 (1.25%)

Всього рядків з пропусками: 25979


In [6]:
# Data Cleaning
start = timeit.default_timer()

# Конвертуємо числові стовпці
numeric_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 
                'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Створюємо стовпець DateTime
df['DateTime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S', errors='coerce')

# Заповнюємо пропуски медіаною (для числових стовпців)
for col in numeric_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

# Додаємо допоміжні стовпці
df['Hour'] = df['DateTime'].dt.hour
df['DayOfWeek'] = df['DateTime'].dt.dayofweek  # 0=Monday, 6=Sunday

elapsed = timeit.default_timer() - start
print(f"Час очищення даних: {elapsed:.2f} секунд")
print(f"Розмір після очищення: {df.shape}")
print(f"Пропусків залишилось: {df[numeric_cols].isnull().sum().sum()}")
df.head()

Час очищення даних: 35.49 секунд
Розмір після очищення: (2075259, 12)
Пропусків залишилось: 0


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,DateTime,Hour,DayOfWeek
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00,17,5
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00,17,5
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00,17,5
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00,17,5
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00,17,5


## 4. Формування вибірок

### 4.1. Записи з загальною активною потужністю > 5 кВт

Обираємо всі записи, у яких Global_active_power перевищує 5 кВт.

In [7]:
def filter_high_power(df, threshold=5.0):
    """Обирає записи з Global_active_power > threshold кВт."""
    start = timeit.default_timer()
    
    result = df[df['Global_active_power'] > threshold]
    
    elapsed = timeit.default_timer() - start
    
    print(f"Записи з активною потужністю > {threshold} кВт:")
    print(f"  Знайдено: {len(result)} записів ({len(result)/len(df)*100:.2f}% від загальної кількості)")
    print(f"  Час виконання: {elapsed:.6f} секунд")
    print(f"  Середня потужність у вибірці: {result['Global_active_power'].mean():.2f} кВт")
    return result

high_power = filter_high_power(df)
high_power.head(10)

Записи з активною потужністю > 5.0 кВт:
  Знайдено: 17547 записів (0.85% від загальної кількості)
  Час виконання: 0.041286 секунд
  Середня потужність у вибірці: 5.85 кВт


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,DateTime,Hour,DayOfWeek
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00,17,5
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00,17,5
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00,17,5
11,16/12/2006,17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0,2006-12-16 17:35:00,17,5
12,16/12/2006,17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0,2006-12-16 17:36:00,17,5
13,16/12/2006,17:37:00,5.268,0.398,232.91,22.6,0.0,2.0,17.0,2006-12-16 17:37:00,17,5
20,16/12/2006,17:44:00,5.894,0.000,232.69,25.4,0.0,0.0,16.0,2006-12-16 17:44:00,17,5
21,16/12/2006,17:45:00,7.706,0.000,230.98,33.2,0.0,0.0,17.0,2006-12-16 17:45:00,17,5
22,16/12/2006,17:46:00,7.026,0.000,232.21,30.6,0.0,0.0,16.0,2006-12-16 17:46:00,17,5
23,16/12/2006,17:47:00,5.174,0.000,234.19,22.0,0.0,0.0,17.0,2006-12-16 17:47:00,17,5


### 4.2. Записи з силою струму 19-20 А, де пральна машина + холодильник > бойлер + кондиціонер

Обираємо записи з Global_intensity в межах 19-20 А, серед них — ті, де Sub_metering_2 (пральна машина, сушарка, холодильник, освітлення) > Sub_metering_3 (бойлер та кондиціонер).

In [8]:
def filter_current_and_consumption(df, current_min=19, current_max=20):
    """
    Обирає записи з силою струму в межах current_min-current_max А,
    серед них — ті, де Sub_metering_2 > Sub_metering_3.
    """
    start = timeit.default_timer()
    
    # Фільтр за силою струму
    current_filter = df[(df['Global_intensity'] >= current_min) & 
                        (df['Global_intensity'] <= current_max)]
    
    # Фільтр: пральна+холодильник > бойлер+кондиціонер
    result = current_filter[current_filter['Sub_metering_2'] > current_filter['Sub_metering_3']]
    
    elapsed = timeit.default_timer() - start
    
    print(f"Фільтрація за силою струму {current_min}-{current_max} А:")
    print(f"  Записів з струмом {current_min}-{current_max} А: {len(current_filter)}")
    print(f"  З них Sub_metering_2 > Sub_metering_3: {len(result)}")
    print(f"  Час виконання: {elapsed:.6f} секунд")
    return result

current_consumption = filter_current_and_consumption(df)
current_consumption.head(10)

Фільтрація за силою струму 19-20 А:
  Записів з струмом 19-20 А: 7021
  З них Sub_metering_2 > Sub_metering_3: 2509
  Час виконання: 0.044286 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,DateTime,Hour,DayOfWeek
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0,2006-12-16 18:09:00,18,5
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0,2006-12-17 01:04:00,1,6
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0,2006-12-17 01:08:00,1,6
475,17/12/2006,01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0,2006-12-17 01:19:00,1,6
476,17/12/2006,01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0,2006-12-17 01:20:00,1,6
477,17/12/2006,01:21:00,4.652,0.142,237.92,19.4,0.0,36.0,0.0,2006-12-17 01:21:00,1,6
508,17/12/2006,01:52:00,4.622,0.240,239.59,19.2,0.0,37.0,0.0,2006-12-17 01:52:00,1,6
944,17/12/2006,09:08:00,4.762,0.088,237.44,20.0,0.0,38.0,0.0,2006-12-17 09:08:00,9,6
945,17/12/2006,09:09:00,4.506,0.088,237.25,19.0,0.0,38.0,0.0,2006-12-17 09:09:00,9,6
1051,17/12/2006,10:55:00,4.444,0.136,235.97,19.2,2.0,50.0,17.0,2006-12-17 10:55:00,10,6


### 4.3. Випадкова вибірка 500000 записів — середнє по 3 групах споживання

In [9]:
def random_sample_means(df, n=500000):
    """
    Обирає випадковим чином n записів (без повторів)
    та обчислює середні величини 3 груп споживання.
    """
    start = timeit.default_timer()
    
    sample = df.sample(n=n, replace=False, random_state=42)
    
    means = {
        'Sub_metering_1 (кухня)': sample['Sub_metering_1'].mean(),
        'Sub_metering_2 (пральна, холодильник, освітлення)': sample['Sub_metering_2'].mean(),
        'Sub_metering_3 (бойлер, кондиціонер)': sample['Sub_metering_3'].mean()
    }
    
    elapsed = timeit.default_timer() - start
    
    print(f"Випадкова вибірка {n} записів (без повторів):")
    print(f"  Час виконання: {elapsed:.6f} секунд")
    print(f"\nСередні величини груп споживання (Вт·год):")
    print("-" * 50)
    for name, mean_val in means.items():
        print(f"  {name}: {mean_val:.4f}")
    
    return sample, means

sample, means = random_sample_means(df)

Випадкова вибірка 500000 записів (без повторів):
  Час виконання: 0.810513 секунд

Середні величини груп споживання (Вт·год):
--------------------------------------------------
  Sub_metering_1 (кухня): 1.1090
  Sub_metering_2 (пральна, холодильник, освітлення): 1.2837
  Sub_metering_3 (бойлер, кондиціонер): 6.3971


### 4.4. Записи після 18:00 з потужністю > 6 кВт, складна вибірка

1. Обираємо записи після 18:00 з Global_active_power > 6 кВт
2. Серед них — ті, де група 2 (Sub_metering_2) найбільша
3. Обираємо кожен 3-й з першої половини та кожен 4-й з другої половини

In [10]:
def complex_evening_filter(df, hour_threshold=18, power_threshold=6.0):
    """
    Складна вибірка:
    1. Після 18:00 з потужністю > 6 кВт (в середньому за хвилину)
    2. Група 2 найбільша
    3. Кожен 3-й з 1-ї половини + кожен 4-й з 2-ї половини
    """
    start = timeit.default_timer()
    
    # Крок 1: після 18:00, потужність > 6 кВт
    evening = df[(df['Hour'] >= hour_threshold) & (df['Global_active_power'] > power_threshold)]
    print(f"Крок 1 — Записи після {hour_threshold}:00 з потужністю > {power_threshold} кВт: {len(evening)}")
    
    # Крок 2: Sub_metering_2 > Sub_metering_1 та Sub_metering_2 > Sub_metering_3
    group2_max = evening[
        (evening['Sub_metering_2'] > evening['Sub_metering_1']) & 
        (evening['Sub_metering_2'] > evening['Sub_metering_3'])
    ]
    print(f"Крок 2 — З них група 2 найбільша: {len(group2_max)}")
    
    # Крок 3: ділимо навпіл
    half = len(group2_max) // 2
    first_half = group2_max.iloc[:half]
    second_half = group2_max.iloc[half:]
    
    # Кожен 3-й з першої половини
    every_3rd = first_half.iloc[::3]
    # Кожен 4-й з другої половини
    every_4th = second_half.iloc[::3]  # ::4 for every 4th
    every_4th = second_half.iloc[::4]
    
    result = pd.concat([every_3rd, every_4th])
    
    elapsed = timeit.default_timer() - start
    
    print(f"Крок 3 — Кожен 3-й з 1-ї половини ({len(every_3rd)}) + кожен 4-й з 2-ї ({len(every_4th)})")
    print(f"  Підсумковий розмір вибірки: {len(result)}")
    print(f"  Час виконання: {elapsed:.6f} секунд")
    return result

complex_result = complex_evening_filter(df)
complex_result.head(10)

Крок 1 — Записи після 18:00 з потужністю > 6.0 кВт: 2882
Крок 2 — З них група 2 найбільша: 1061
Крок 3 — Кожен 3-й з 1-ї половини (177) + кожен 4-й з 2-ї (133)
  Підсумковий розмір вибірки: 310
  Час виконання: 0.044869 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,DateTime,Hour,DayOfWeek
41,16/12/2006,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0,2006-12-16 18:05:00,18,5
44,16/12/2006,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0,2006-12-16 18:08:00,18,5
17494,28/12/2006,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0,2006-12-28 20:58:00,20,3
17498,28/12/2006,21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0,2006-12-28 21:02:00,21,3
17501,28/12/2006,21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0,2006-12-28 21:05:00,21,3
17504,28/12/2006,21:08:00,7.352,0.000,235.45,31.2,1.0,73.0,17.0,2006-12-28 21:08:00,21,3
17507,28/12/2006,21:11:00,9.048,0.000,231.48,39.0,34.0,71.0,16.0,2006-12-28 21:11:00,21,3
17510,28/12/2006,21:14:00,9.118,0.108,231.18,39.4,36.0,72.0,16.0,2006-12-28 21:14:00,21,3
17513,28/12/2006,21:17:00,7.040,0.130,233.27,30.2,37.0,38.0,17.0,2006-12-28 21:17:00,21,3
18952,29/12/2006,21:16:00,6.146,0.116,230.53,26.6,0.0,70.0,0.0,2006-12-29 21:16:00,21,4


## 5. Нормалізація та стандартизація

**Нормалізація (Min-Max):** переводить значення в діапазон [0, 1]

$$x_{norm} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

**Стандартизація (Z-score):** центрує дані навколо 0 з σ=1

$$x_{std} = \frac{x - \mu}{\sigma}$$

In [11]:
# Нормалізація та стандартизація
start = timeit.default_timer()

numeric_cols_to_normalize = ['Global_active_power', 'Global_reactive_power', 'Voltage', 
                              'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']

df_normalized = df.copy()
df_standardized = df.copy()

for col in numeric_cols_to_normalize:
    # Min-Max нормалізація
    col_min = df[col].min()
    col_max = df[col].max()
    if col_max != col_min:
        df_normalized[col] = (df[col] - col_min) / (col_max - col_min)
    
    # Z-score стандартизація
    col_mean = df[col].mean()
    col_std = df[col].std()
    if col_std != 0:
        df_standardized[col] = (df[col] - col_mean) / col_std

elapsed = timeit.default_timer() - start
print(f"Час нормалізації/стандартизації: {elapsed:.4f} секунд")

print("\nНормалізовані дані (Min-Max) — перші 5 записів:")
df_normalized[numeric_cols_to_normalize].head()

Час нормалізації/стандартизації: 3.5470 секунд

Нормалізовані дані (Min-Max) — перші 5 записів:


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,0.374796,0.300719,0.376090,0.377593,0.0,0.0125,0.548387
1,0.478363,0.313669,0.336995,0.473029,0.0,0.0125,0.516129
2,0.479631,0.358273,0.326010,0.473029,0.0,0.0250,0.548387
3,0.480898,0.361151,0.340549,0.473029,0.0,0.0125,0.548387
4,0.325005,0.379856,0.403231,0.323651,0.0,0.0125,0.548387


In [12]:
print("Стандартизовані дані (Z-score) — перші 5 записів:")
df_standardized[numeric_cols_to_normalize].head()

Стандартизовані дані (Z-score) — перші 5 записів:


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2.975591,2.629138,-1.864146,3.120053,-0.181154,-0.048773,1.262163
1,4.062976,2.789788,-2.239957,4.160249,-0.181154,-0.048773,1.143202
2,4.076283,3.343135,-2.345557,4.160249,-0.181154,0.124020,1.262163
3,4.089590,3.378835,-2.205793,4.160249,-0.181154,-0.048773,1.262163
4,2.452809,3.610884,-1.603252,2.532116,-0.181154,-0.048773,1.262163


In [13]:
# Перевірка: після стандартизації середнє ≈ 0, std ≈ 1
print("Перевірка стандартизації (середнє та стд. відхилення):")
print("-" * 50)
for col in numeric_cols_to_normalize:
    print(f"  {col:30s}: μ={df_standardized[col].mean():.6f}, σ={df_standardized[col].std():.6f}")

Перевірка стандартизації (середнє та стд. відхилення):
--------------------------------------------------
  Global_active_power           : μ=-0.000000, σ=1.000000
  Global_reactive_power         : μ=0.000000, σ=1.000000
  Voltage                       : μ=-0.000000, σ=1.000000
  Global_intensity              : μ=0.000000, σ=1.000000
  Sub_metering_1                : μ=0.000000, σ=1.000000
  Sub_metering_2                : μ=0.000000, σ=1.000000
  Sub_metering_3                : μ=-0.000000, σ=1.000000


## 6. Коефіцієнти кореляції Пірсона та Спірмена

Обчислюємо кореляцію між **Global_active_power** (загальна активна потужність) та **Global_intensity** (сила струму) — двома числовими атрибутами.

In [14]:
start = timeit.default_timer()

# Коефіцієнт Пірсона
pearson_corr = df['Global_active_power'].corr(df['Global_intensity'], method='pearson')

# Коефіцієнт Спірмена
spearman_corr = df['Global_active_power'].corr(df['Global_intensity'], method='spearman')

elapsed = timeit.default_timer() - start

print("Кореляція між Global_active_power та Global_intensity:")
print("=" * 50)
print(f"  Коефіцієнт Пірсона:  {pearson_corr:.6f}")
print(f"  Коефіцієнт Спірмена: {spearman_corr:.6f}")
print(f"  Час обчислення: {elapsed:.6f} секунд")
print()

if abs(pearson_corr) > 0.7:
    print("  → Сильна кореляція (за модулем > 0.7)")
elif abs(pearson_corr) > 0.4:
    print("  → Помірна кореляція (за модулем 0.4-0.7)")
else:
    print("  → Слабка кореляція (за модулем < 0.4)")

Кореляція між Global_active_power та Global_intensity:
  Коефіцієнт Пірсона:  0.998891
  Коефіцієнт Спірмена: 0.995508
  Час обчислення: 15.920662 секунд

  → Сильна кореляція (за модулем > 0.7)


In [15]:
# Матриця кореляцій для всіх числових стовпців
print("Матриця кореляцій Пірсона:")
corr_matrix = df[numeric_cols_to_normalize].corr(method='pearson').round(3)
corr_matrix

Матриця кореляцій Пірсона:


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
Global_active_power,1.000,0.248,-0.400,0.999,0.485,0.435,0.640
Global_reactive_power,0.248,1.000,-0.112,0.267,0.124,0.140,0.091
Voltage,-0.400,-0.112,1.000,-0.411,-0.196,-0.167,-0.268
Global_intensity,0.999,0.267,-0.411,1.000,0.490,0.441,0.628
Sub_metering_1,0.485,0.124,-0.196,0.490,1.000,0.055,0.104
Sub_metering_2,0.435,0.140,-0.167,0.441,0.055,1.000,0.082
Sub_metering_3,0.640,0.091,-0.268,0.628,0.104,0.082,1.000


## 7. One Hot Encoding категоріального атрибута

Створюємо категоріальний атрибут — **частина доби** (ранок, день, вечір, ніч) на основі стовпця Hour, та виконуємо One Hot Encoding.

In [16]:
start = timeit.default_timer()

# Створюємо категоріальний атрибут "Частина доби"
def get_day_part(hour):
    if 6 <= hour < 12:
        return 'morning'
    elif 12 <= hour < 18:
        return 'afternoon'
    elif 18 <= hour < 23:
        return 'evening'
    else:
        return 'night'

df['DayPart'] = df['Hour'].apply(get_day_part)

print("Розподіл за частинами доби:")
print(df['DayPart'].value_counts())

# One Hot Encoding
df_encoded = pd.get_dummies(df, columns=['DayPart'], prefix='DayPart', dtype=int)

elapsed = timeit.default_timer() - start

print(f"\nЧас виконання OHE: {elapsed:.4f} секунд")
print(f"\nНові стовпці після One Hot Encoding:")
ohe_cols = [col for col in df_encoded.columns if 'DayPart_' in col]
print(f"  {ohe_cols}")
print(f"\nПриклад (перші 10 записів):")
df_encoded[['DateTime', 'Global_active_power'] + ohe_cols].sample(10, random_state=42)

Розподіл за частинами доби:
DayPart
night        605220
afternoon    518796
morning      518760
evening      432483
Name: count, dtype: int64

Час виконання OHE: 3.0614 секунд

Нові стовпці після One Hot Encoding:
  ['DayPart_afternoon', 'DayPart_evening', 'DayPart_morning', 'DayPart_night']

Приклад (перші 10 записів):


,DateTime,Global_active_power,DayPart_afternoon,DayPart_evening,DayPart_morning,DayPart_night
1870606,2010-07-07 18:10:00,0.256,0,1,0,0
213926,2007-05-14 06:50:00,0.466,0,0,1,0
409006,2007-09-26 18:10:00,0.758,0,1,0,0
265806,2007-06-19 07:30:00,1.290,0,0,1,0
1786279,2010-05-10 04:43:00,0.428,0,0,0,1
1397790,2009-08-13 09:54:00,0.602,0,0,1,0
2006309,2010-10-09 23:53:00,1.206,0,0,0,1
365764,2007-08-27 17:28:00,0.182,1,0,0,0
1111484,2009-01-26 14:08:00,1.792,1,0,0,0
1473916,2009-10-05 06:40:00,0.474,0,0,1,0


## Підсумок

У цій частині лабораторної роботи було виконано:
1. Завантажено та очищено датасет електроспоживання домогосподарства
2. Сформовано вибірки з різними умовами фільтрації
3. Проведено нормалізацію (Min-Max) та стандартизацію (Z-score)
4. Обчислено коефіцієнти кореляції Пірсона та Спірмена
5. Виконано One Hot Encoding категоріального атрибута
6. Для кожної операції виконано профілювання часу за допомогою timeit